# Data Preparation

Modify the data so ML algorithms can properly learn from it.

The majority of reasons why we apply some of the changes are described in the data understanding notebook. If not, we explain it here.

### Imports

In [1]:
# Import necessary libraries, functions, objects...

import pandas as pd
import numpy as np

### Load dataset

In [2]:
df = pd.read_csv('../data/bank_term_deposit/bank_term_deposit.csv')
df.head()

,id,split,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,1,labeled,31,management,married,tertiary,no,460,no,no,cellular,28,aug,13,14,-1,0,unknown,no
1,2,labeled,34,blue-collar,married,secondary,no,1826,yes,no,unknown,20,may,203,2,-1,0,unknown,no
2,3,labeled,50,blue-collar,married,secondary,no,290,yes,no,cellular,7,aug,226,3,-1,0,unknown,no
3,4,labeled,42,admin.,divorced,secondary,no,1077,yes,no,unknown,14,may,213,1,-1,0,unknown,no
4,5,labeled,47,services,single,secondary,no,41,yes,no,cellular,5,may,298,1,-1,0,unknown,no


## Select Data

After the previous exploration you may decide to use or not use some of the data sets.

For this exercise there is **no decision to make, you use the only data set we have**.

## Clean Data

### Remove unnecessary features (if any)

### Deal with null or erroneous values (if any)

It would be nice to have a column that indicates when the `pdays` value is null (i.e., -1).

For the other columns containing 'unknown', we'll treat 'unknown' as just another valid category.

Since `pdays` and `poutcome` have their 'unknown' entries in the same rows, handling 'unknown' in `poutcome` as a special category already captures the cases where `pdays` is null. Therefore, no additional processing is needed for `pdays`.

In [3]:
# Convert ? to unknown

df.loc[df['job'] == '?', 'job'] = 'unknown'

### Deal with duplicated rows that are errors (if any)

No duplicates to remove.

### Decide what to do with outliers (if any)

Normally during this step you may decide to remove or change some outliers. However, **for this exercise do not remove any outliers**, leave them as they are.

## Construct Data

Decide if you want to create new features from the existing ones. You can be as creative as you want.

In [4]:
df.head()

,id,split,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,1,labeled,31,management,married,tertiary,no,460,no,no,cellular,28,aug,13,14,-1,0,unknown,no
1,2,labeled,34,blue-collar,married,secondary,no,1826,yes,no,unknown,20,may,203,2,-1,0,unknown,no
2,3,labeled,50,blue-collar,married,secondary,no,290,yes,no,cellular,7,aug,226,3,-1,0,unknown,no
3,4,labeled,42,admin.,divorced,secondary,no,1077,yes,no,unknown,14,may,213,1,-1,0,unknown,no
4,5,labeled,47,services,single,secondary,no,41,yes,no,cellular,5,may,298,1,-1,0,unknown,no


In [5]:
# Temporal

# Convert month names to numbers for easier analysis
month_map = {'jan':1,'feb':2,'mar':3,'apr':4,'may':5,'jun':6,
             'jul':7,'aug':8,'sep':9,'oct':10,'nov':11,'dec':12}
df['month_num'] = df['month'].map(month_map)

# Create a season feature
def month_to_season(m):
    if m in [12, 1, 2]: return 'winter'
    elif m in [3, 4, 5]: return 'spring'
    elif m in [6, 7, 8]: return 'summer'
    return 'autumn'

df['season'] = df['month_num'].apply(month_to_season)

# Cyclical encoding
df['month_sin'] = np.sin(2 * np.pi * df['month_num'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month_num'] / 12)
df['day_sin']   = np.sin(2 * np.pi * df['day'] / 31)
df['day_cos']   = np.cos(2 * np.pi * df['day'] / 31)

# Whether contact happened early or late in the month
df['early_month'] = (df['day'] <= 10).astype(int)
df['late_month']  = (df['day'] >= 20).astype(int)

In [6]:
# Contact

# Average time per contact during current campaign
df['avg_contact_duration'] = np.where(df['campaign'] > 0,
                                      df['duration'] / df['campaign'],
                                      0)

# Contacts per month position
df['contacts_per_month'] = df['campaign'] / df['month_num']

In [7]:
# Create a "financial stress" index using balance and loan information
df['financial_stress'] = (
    (df['balance'] < 0).astype(int)  # negative balance
    + df['loan'].map({'yes': 1, 'no': 0})
    + df['default'].map({'yes': 1, 'no': 0})
)

## Integrate Data

Decide if you want to integrate data from other sources.

**This is not needed for the exercises.**

## Feature Engineering

In [8]:
df.head()

,id,split,age,job,marital,education,default,balance,housing,loan,...,season,month_sin,month_cos,day_sin,day_cos,early_month,late_month,avg_contact_duration,contacts_per_month,financial_stress
0,1,labeled,31,management,married,tertiary,no,460,no,no,...,summer,-0.866025,-0.500000,-0.571268,0.820763,0,1,0.928571,1.750,0
1,2,labeled,34,blue-collar,married,secondary,no,1826,yes,no,...,spring,0.500000,-0.866025,-0.790776,-0.612106,0,1,101.500000,0.400,0
2,3,labeled,50,blue-collar,married,secondary,no,290,yes,no,...,summer,-0.866025,-0.500000,0.988468,0.151428,1,0,75.333333,0.375,0
3,4,labeled,42,admin.,divorced,secondary,no,1077,yes,no,...,spring,0.500000,-0.866025,0.299363,-0.954139,0,0,213.000000,0.200,0
4,5,labeled,47,services,single,secondary,no,41,yes,no,...,spring,0.500000,-0.866025,0.848644,0.528964,1,0,298.000000,0.200,0


### Integer

Convert some "string boolean" columns to integer.

In [9]:
for col in ['default', 'housing', 'loan', 'y']:
    df[col] = (df[col] == 'yes').astype(int)

### Encoding

Apply the encodings that you consider more appropriate for the categorical variables.

In [10]:
# Object columns
for col in df.select_dtypes('object').columns:
    print(col)
    print(df[col].unique())
    print()

split
<StringArray>
['labeled', 'leaderboard']
Length: 2, dtype: str

job
<StringArray>
[   'management',   'blue-collar',        'admin.',      'services',
    'technician',    'unemployed',       'retired',       'unknown',
  'entrepreneur', 'self-employed',       'student',     'housemaid']
Length: 12, dtype: str

marital
<StringArray>
['married', 'divorced', 'single']
Length: 3, dtype: str

education
<StringArray>
['tertiary', 'secondary', 'unknown', 'primary']
Length: 4, dtype: str

contact
<StringArray>
['cellular', 'unknown', 'telephone']
Length: 3, dtype: str

month
<StringArray>
['aug', 'may', 'jun', 'apr', 'nov', 'jan', 'feb', 'jul', 'oct', 'sep', 'mar',
 'dec']
Length: 12, dtype: str

poutcome
<StringArray>
['unknown', 'failure', 'other', 'success']
Length: 4, dtype: str

season
<StringArray>
['summer', 'spring', 'autumn', 'winter']
Length: 4, dtype: str



C:\Users\Alumne_mati1\AppData\Local\Temp\ipykernel_24556\54084117.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes('object').columns:


#### Ordinal Encoding

Do not include `month` as we already computed `month_num`.

In [11]:
categories = {
    'education': ['unknown', 'primary', 'secondary', 'tertiary'],
}

In [12]:
# Special column for unknown education
df['education_unknown'] = (df['education'] == 0).astype(int)

#### One-hot encoding

In [13]:
fts_to_onehot = [
    'job',
    'marital',
    'contact',
    'poutcome',
    'season'
]

In [14]:
# Number of unique elements, to make sure we don't create too many columns
for ft in fts_to_onehot:
    print(ft, df[ft].nunique())

job 12
marital 3
contact 3
poutcome 4
season 4


### Binning

Apply binning to some columns if you consider it appropriate.

In [15]:
age_bins = [0, 25, 35, 45, 55, 65, 100]
age_labels = range(len(age_bins) - 1)

df['age_group'] = pd.cut(df['age'], bins=age_bins, labels=age_labels, right=False).astype(int)

In [16]:
df['balance'].describe()

count     3063.000000
mean      1313.493960
std       2768.188907
min      -2076.000000
25%         58.000000
50%        394.000000
75%       1334.500000
max      42042.000000
Name: balance, dtype: float64

In [17]:
balance_bins = [-np.inf, -1000, 0, 1000, 5000, 10000, np.inf]
balance_labels = range(len(balance_bins) - 1)

df['balance_group'] = pd.cut(df['balance'], bins=balance_bins, labels=balance_labels).astype(int)

### Remove unnecessary features

In [18]:
df.drop(columns='month', inplace=True)

## Save New Dataset

In [19]:
df.to_csv('../data/bank_term_deposit/bank_term_deposit_prepared.csv', index=False)

In [20]:
dp = pd.read_csv('../data/bank_term_deposit/bank_term_deposit_prepared.csv')
dp.shape

(3063, 32)